# SBM's own pooled weights

The pooled weights in `multi_pair_models.ipynb` were fitted on ER, PAPER and Chung–Lu pairs, which have no communities. Instead of asking whether those weights transfer to SBMs, this notebook collects the same kind of data on SBM pairs and fits SBM's own weights. It then asks:
1. Do the same features matter once the vertices form communities? (SBM's weights next to the other models')
2. Does SBM's own score rank seed sets, and does best of N with it beat random on new pairs?
3. Does where the seeds sit among the communities matter beyond the four features?

**Graphs.** The two main settings of `sbm_rules.ipynb`, with fresh pairs from their own seed stream:

| Setting | What it is | Matched round-1 model |
|---|---|---|
| SBM | 4 equal blocks of 150 vertices, 80% of edges inside blocks, otherwise ER | ER |
| Degree-corrected SBM | The same blocks on Chung–Lu weights | Chung–Lu |

Mean degree, ρ and GOAT's regulariser are those of the matched model.

**Budget.** One per algorithm and setting: k50 from `sbm_rules.ipynb`'s calibration, where a random set succeeds about half the time. This is how the multi-pair collection's budgets were chosen. If a calibration is missing, it runs first here and is shared with `sbm_rules.ipynb`.

**Seed sets and runs,** exactly as in the multi-pair collection. Each pair has 500 sets:
- **Random (300):** the first 300 of a 20,000-set random pool.
- **Tail (100):** from the top 5% of the pool by degree, pair mass, uniqueness, and their sum.
- **Contrast (100):** 50 degree-matched pairs, low against high pair mass or uniqueness.

Each set is run once with round 1's solver settings. Every set also records the largest block's share of its seeds: 0.25 means spread evenly over the 4 blocks, 1 means all in one block. The share comes from the true block labels, so it is a diagnostic: it says whether community placement matters, not something one could compute from graph A alone. The saved weights use only the four core features.

**Direct test (section 7).** After the weights are fitted and saved, best of 100 and best of 1,000 are run with them on `sbm_rules.ipynb`'s 20 test pairs per setting, which were never used for fitting. They choose from the same 1,000 random draws as that notebook's best of 1,000 with the averaged weights, and meet the same solver seeds, so the comparison isolates the weights. `sbm_rules.ipynb` must have run first.

**Pairs per setting:** percolation 30, GOAT 20, FAQ 16. **Run time:** about 11 hours with 3 jobs for the collection, so best overnight, plus about 25 minutes for the direct test. Rows are written set by set, and rerunning resumes.

| Section | Content |
|---|---|
| 1 | The data |
| 2 | SBM's weights next to the other models' |
| 3 | Ranking within pairs (AUC) |
| 4 | Success by held-out score percentile (a preview of best of N) |
| 5 | Does where the seeds sit among the blocks matter? |
| 6 | Saving SBM's weights |
| 7 | Direct test: best of N with SBM's weights on new pairs |
| 8 | Letting the score pick among round 1's rules |

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'sgm_matplotlib_cache'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

EXPERIMENT_DIR = Path.cwd().resolve()
if EXPERIMENT_DIR.name != 'fixed_pair_seed_set_classification':
    EXPERIMENT_DIR = EXPERIMENT_DIR / 'fixed_pair_seed_set_classification'
if str(EXPERIMENT_DIR.parent) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR.parent))

import importlib

from selection_rule_comparison.archive import sbm_graphs as G
from selection_rule_comparison.archive import sbm_rules as SR
from fixed_pair_seed_set_classification import sbm_collection as C

G = importlib.reload(G)
SR = importlib.reload(SR)
C = importlib.reload(C)     # picks up edits to sbm_collection.py without restarting the kernel

pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', 120)

RUN_COLLECTION = True                                 # False: only analyse saved results
RUN_SELECTION_TEST = True                             # section 7: best of N with SBM's weights (about 25 min)
ALGORITHMS_TO_RUN = ['percolation', 'goat', 'faq']    # remove any to skip it
SETTINGS_TO_RUN = ['sbm', 'dcsbm']
PAIRS = {'percolation': 30, 'goat': 20, 'faq': 16}    # pairs per setting; more can be added later
JOBS = 3                                              # tasks run in parallel

## Run

The cell below prints an estimate per algorithm, then one line per finished pair (and per budget calibration, if one is still missing). Pairs run in order of pair index across settings, so stopping early leaves every setting with a similar number of pairs.

The interrupt button stops the workers too. To run in a terminal instead, from the repository root: `python -m fixed_pair_seed_set_classification.sbm_collection`. Ctrl-C stops it, and `--status` shows how far it is. Then set `RUN_COLLECTION = False` here to analyse.

In [ ]:
if RUN_COLLECTION:
    C.run_all(ALGORITHMS_TO_RUN, SETTINGS_TO_RUN, n_pairs=PAIRS, jobs=JOBS)
display(C.status(settings=SETTINGS_TO_RUN, n_pairs=PAIRS))

## 1. The data

Per algorithm and setting:
- **Budget:** k50 from the calibration.
- **Random success:** the mean over pairs, and the lowest and highest pair. Pairs where random sets always or never succeed say nothing about which sets are better.
- **Pairs with both outcomes:** the pairs the weights are fitted on.
- **Largest block share:** the random sets' average. Seeds spread perfectly over 4 blocks would give 0.25.

In [ ]:
data = C.load_sets(settings=SETTINGS_TO_RUN)
display(C.overview(data))

## 2. SBM's weights next to the other models'

The pooled model is fitted as in `multi_pair_models.ipynb`. It is a logistic regression of success on the four core features, each in SDs of the pair's random sets, with its own intercept per pair. The random and tail sets are used. A weight is the change in log-odds of success per SD of the feature, holding the others fixed.

In the figure:
- **Open markers:** ER (blue circle), PAPER (orange triangle) and Chung–Lu (aqua square), as saved from the multi-pair collection.
- **Filled markers:** SBM (blue, matched to ER) and degree-corrected SBM (aqua, matched to Chung–Lu), fitted here.
- **Whiskers:** 95% intervals from resampling whole pairs.

If a filled marker sits near its open partner, communities leave that feature's role unchanged. The table gives the numbers.

In [ ]:
weights = C.weights_table(data)
C.plot_weights(weights)
plt.show()
display(C.show_weights(weights))

## 3. Ranking within pairs

AUC on each pair's 300 random sets: the chance that a successful set scores higher than a failed one, averaged over pairs where random sets both succeed and fail. 0.5 is chance. The scores compared:
- **Own score (held out):** SBM's weights, each pair scored by a model fitted without it (5-fold, grouped by pair).
- **Own score + block share:** the same with the largest block's share as a fifth feature.
- **Degree alone:** seed-to-nonseed edges.
- **Matched model's weights:** ER's for SBM, Chung–Lu's for degree-corrected SBM, unchanged.
- **Weights averaged:** the ER, PAPER and Chung–Lu weights averaged, as used by best of 1,000 in `sbm_rules.ipynb`.

The intervals are 95% intervals over pairs. The second table gives, for comparison, the same held-out AUC on the multi-pair data of the round-1 models.

In [ ]:
ranking, scored = C.ranking_table(data)
display(ranking.drop(columns='auc_mean'))
reference = C.reference_auc(ALGORITHMS_TO_RUN)
display(reference if not reference.empty else 'No multi-pair data found for comparison.')

## 4. Success by held-out score percentile

A preview of best of N, on the collection's own pairs. The success rate of held-out random and tail sets, binned by their score's percentile among the pair's 20,000 random sets. The score uses SBM's own weights fitted without the pair. The sets in the top bins were not chosen by this score (the tail sets were chosen by single features), so section 7 tests the actual best-of-N picks on new pairs.
- **Solid lines:** blue for SBM, aqua for degree-corrected SBM.
- **Dotted lines:** the average random set, in the same colour.

The top two bins are where best of 100 and best of 1,000 pick. A curve that rises to the right means best of N with SBM's own score beats random.

In [ ]:
percentiles = C.success_by_percentile(scored)
C.plot_percentile_success(percentiles)
plt.show()
display(percentiles.assign(algorithm=percentiles['algorithm'].map(C.ALGORITHM_LABELS),
                           setting=percentiles['setting'].map(C.SETTING_LABELS))
        .set_index(['algorithm', 'setting', 'percentile'])[['sets', 'success']].round(3))

## 5. Does where the seeds sit among the blocks matter?

- **First table:** the random sets' success by the largest block's share of their seeds, from the most spread to the most concentrated. With few seeds there are only a few possible shares, and each gets its own row. Otherwise the shares are grouped into about four bins. The intervals are 95% intervals from resampling pairs.
- **Second table:** the pooled weights with the largest block's share added as a fifth feature (per SD, 95% intervals). A negative weight on the share means that concentrating the seeds in one block hurts at equal degree, harmonic, pair mass and uniqueness. If the other weights barely move, the share adds something they do not capture.
- **Third table:** the degree-matched contrasts, as in `multi_pair_models.ipynb`. Each contrast is a pair of sets with the same number of seed-to-nonseed edges but low against high pair mass or uniqueness. It counts the contrasts where only the high set succeeded against those where only the low set did, with a sign test.

In [ ]:
display(C.block_share_table(data))
display(C.five_feature_weights(data))
display(C.contrast_table(data))

## 6. Saving SBM's weights

SBM's pooled weights (the four core features only, all computed from graph A), in the format of the multi-pair `pooled_weights.json`. They are saved to `data/sbm_multi_pair/pooled_weights.json`, where section 7 reads them.

In [ ]:
path = C.save_weights(weights)
print(f'Saved to {path}')

## 7. Direct test: best of N with SBM's weights on new pairs

On `sbm_rules.ipynb`'s 20 test pairs per setting and its budget grid (around k50), on each pair and budget:
- **Best of 100 and best of 1,000 (SBM weights):** the highest-scoring set by SBM's own weights among the first 100 and among all 1,000 random draws.
- **Compared with,** on the same pairs and budgets: random, top degree, best of 1,000 by degree alone, and best of 1,000 by the averaged ER, PAPER and Chung–Lu weights.

The 1,000 draws are the same ones the averaged-weight best of 1,000 chose from, which the run checks, and the solver seeds are the same. So best of 1,000 with SBM's weights against best of 1,000 with the averaged weights differs only in the weights. A set that `sbm_rules.ipynb` already ran at that budget is not run again: its stored result is the same run.

If SBM's weights are refitted later (for example on more pairs), the run refuses to mix results; move the `sbm_weights` folders away to rerun.

In [ ]:
if RUN_SELECTION_TEST:
    SR.run_sbm_weights(ALGORITHMS_TO_RUN, SETTINGS_TO_RUN, jobs=JOBS)
display(SR.sbm_weights_status(ALGORITHMS_TO_RUN, SETTINGS_TO_RUN))

**Results.**
- **Figure:** the share of the 20 pairs that succeed, against the budget. Grey is random and black is top degree. Aqua is best of 1,000 by degree alone, and solid dark blue is best of 1,000 by the averaged weights. Dashed blue with open diamonds is best of 100 (light) and best of 1,000 (dark) by SBM's own weights.
- **First table:** mean success over the budget grid and mean queries needed, with 95% intervals over pairs.
- **Second table:** pair by pair, best of N with SBM's weights against random, the averaged weights, degree alone and top degree. It gives queries needed (fewer / same / more, with a sign test) and the change in mean success.

SBM's weights pass the direct test if best of N with them beats random clearly. Beating the averaged weights and degree alone shows that fitting on SBM was worth it.

In [ ]:
ev = SR.load_evaluations(ALGORITHMS_TO_RUN, SETTINGS_TO_RUN)
if (ev['rule'].isin(SR.SBM_RULES)).any():
    queries = SR.queries_needed(ev)
    curves = SR.success_curves(ev)
    SR.plot_success_curves(curves[curves['rule'].isin(SR.SELECTION_RULES)])
    plt.show()
    display(SR.selection_summary(ev, queries))
    display(SR.selection_comparison(ev, queries))
else:
    print('No results for best of N with SBM weights yet.')

## 8. Letting the score pick among round 1's rules

On round 1's pairs, choosing on each pair and budget whichever round-1 rule's set had the highest pooled score did about as well as the best rule chosen in hindsight. This checks the same idea on the SBM test pairs, with the averaged weights and with SBM's own weights. Top degree by block is left out of the candidates, because it uses the true block labels.
- **Columns:** mean success over the budget grid for top degree, for the rule picked by each score, and for the best fixed rule in hindsight.
- **Differences:** pair by pair with 95% intervals, and the rule each score picked most often.

In [ ]:
display(SR.score_picked_table(ev, SR.read_sbm_scores(ALGORITHMS_TO_RUN, SETTINGS_TO_RUN)))